<a href="https://colab.research.google.com/github/kej534923-maker/ECON5200-Applied-Data-Analytics/blob/main/Lab13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

In [2]:

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/kej534923-maker/ECON5200-Applied-Data-Analytics/refs/heads/main/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols('Sale_Price ~ Property_Age', data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        18:29:54   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [3]:
df

,Property_Age,Distance_to_Tech_Hub,Sale_Price
0,77.5,38.1,684100.56
1,11.0,95.1,413634.22
2,47.7,73.5,456709.35
3,61.9,60.3,624533.95
4,100.8,16.4,870137.54
...,...,...,...
995,87.7,10.1,932592.35
996,21.2,91.8,412741.12
997,96.5,14.5,880901.56
998,20.1,95.1,396659.79


In [11]:
# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.954
Method:                 Least Squares   F-statistic:                 1.040e+04
Date:                Fri, 13 Mar 2026   Prob (F-statistic):               0.00
Time:                        18:40:20   Log-Likelihood:                -11982.
No. Observations:                1000   AIC:                         2.397e+04
Df Residuals:                     997   BIC:                         2.399e+04
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             1.203e+06 

In [12]:
# Step 3: FWL Theorem Manual Proof
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid
# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals~Age_Residuals-1', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])




FWL Isolated Age Coefficient: -2063.129216802139


In [15]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import plotly.graph_objects as go

# ── 1. Synthetic Data (replace with your actual DataFrame) ───────────────────
np.random.seed(42)
n = 200
Property_Age       = np.random.uniform(1, 50, n)
Distance_to_Tech   = np.random.uniform(0.5, 30, n)
Sale_Price         = (500_000
                      - 3_000  * Property_Age
                      - 8_000  * Distance_to_Tech
                      + np.random.normal(0, 20_000, n))

df = pd.DataFrame({
    "Sale_Price":         Sale_Price,
    "Property_Age":       Property_Age,
    "Distance_to_Tech_Hub": Distance_to_Tech
})

# ── 2. OLS Regression ────────────────────────────────────────────────────────
X = sm.add_constant(df[["Property_Age", "Distance_to_Tech_Hub"]])  # adds intercept
y = df["Sale_Price"]

model  = sm.OLS(y, X).fit()
print(model.summary())

# ── 3. Extract Coefficients from statsmodels Results ────────────────────────
# model.params is a pandas Series indexed by variable name:
#   Index 0 → "const"          (β₀ intercept)
#   Index 1 → "Property_Age"   (β₁)
#   Index 2 → "Distance_to_Tech_Hub" (β₂)
intercept   = model.params["const"]
coef_age    = model.params["Property_Age"]
coef_dist   = model.params["Distance_to_Tech_Hub"]

# ── 4. Build the Meshgrid for the Regression Hyperplane ─────────────────────
# np.linspace creates evenly spaced values spanning the observed range of each
# predictor — this ensures the plane covers (but doesn't extrapolate beyond)
# the data cloud.
age_range  = np.linspace(df["Property_Age"].min(),
                         df["Property_Age"].max(), 40)        # 40 pts on X-axis
dist_range = np.linspace(df["Distance_to_Tech_Hub"].min(),
                         df["Distance_to_Tech_Hub"].max(), 40) # 40 pts on Y-axis

# np.meshgrid converts the two 1-D arrays into two 2-D grids (shape 40×40).
# Every (i, j) cell represents one (age, distance) combination in the plane.
age_grid, dist_grid = np.meshgrid(age_range, dist_range)

# Apply the OLS equation at every grid point:
#   ŷ = β₀ + β₁·age + β₂·distance
# This produces a 40×40 matrix of predicted Sale_Prices — the hyperplane surface.
price_grid = intercept + coef_age * age_grid + coef_dist * dist_grid

# ── 5. Build the Plotly Figure ───────────────────────────────────────────────
fig = go.Figure()

# — Scatter: actual data points —
fig.add_trace(go.Scatter3d(
    x=df["Property_Age"],
    y=df["Distance_to_Tech_Hub"],
    z=df["Sale_Price"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["Sale_Price"],        # colour by sale price for extra depth
        colorscale="Viridis",
        opacity=0.75,
        colorbar=dict(title="Sale Price ($)", x=1.02)
    ),
    name="Observed Data"
))

# — Surface: OLS regression hyperplane —
fig.add_trace(go.Surface(
    x=age_grid,
    y=dist_grid,
    z=price_grid,
    colorscale="RdBu",
    opacity=0.55,
    showscale=False,
    name="Regression Hyperplane",
    hovertemplate=(
        "Age: %{x:.1f} yrs<br>"
        "Distance: %{y:.1f} km<br>"
        "Predicted Price: $%{z:,.0f}<extra></extra>"
    )
))

# ── 6. Layout Polish ─────────────────────────────────────────────────────────
r2   = model.rsquared
rmse = np.sqrt(model.mse_resid)

fig.update_layout(
    title=dict(
        text=(f"Multivariate OLS — Sale Price | "
              f"R² = {r2:.3f} | RMSE = ${rmse:,.0f}"),
        x=0.5
    ),
    scene=dict(
        xaxis_title="Property Age (yrs)",
        yaxis_title="Distance to Tech Hub (km)",
        zaxis_title="Sale Price ($)",
        camera=dict(eye=dict(x=1.6, y=-1.6, z=0.8))
    ),
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(x=0.01, y=0.95)
)

fig.show()

# ── 7. Print coefficient summary for quick reference ────────────────────────
print(f"\n{'─'*45}")
print(f"  Intercept (β₀):           ${intercept:>12,.0f}")
print(f"  Property_Age (β₁):        ${coef_age:>12,.0f} per year")
print(f"  Distance_to_Tech_Hub (β₂):${coef_dist:>12,.0f} per km")
print(f"{'─'*45}")

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.945
Model:                            OLS   Adj. R-squared:                  0.944
Method:                 Least Squares   F-statistic:                     1680.
Date:                Fri, 13 Mar 2026   Prob (F-statistic):          1.63e-124
Time:                        19:00:09   Log-Likelihood:                -2261.2
No. Observations:                 200   AIC:                             4528.
Df Residuals:                     197   BIC:                             4538.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                 5.017e+05 


─────────────────────────────────────────────
  Intercept (β₀):           $     501,697
  Property_Age (β₁):        $      -2,996 per year
  Distance_to_Tech_Hub (β₂):$      -8,122 per km
─────────────────────────────────────────────
